### Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    '../data/wustl_iiot_2021.csv',
    parse_dates=['StartTime', 'LastTime']
)

According to the documentation, remove these columns: 'StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'sIpId', 'dIpId', as they are unique to the attacks and would expose the type of the attack to the model.

But we will probably need the addresses to create a graph structure?

In [3]:
df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 12:23:28,2019-08-19 12:23:28,192.168.0.20,192.168.0.2,0,59034,502,10,8,18,...,24,20,44,0.001176,0.053037,0,0.000000,0.0,normal,0
1,2019-08-19 15:13:24,2019-08-19 15:13:24,192.168.0.20,192.168.0.2,0,55841,502,10,8,18,...,24,20,44,0.001308,0.052681,0,0.000000,0.0,normal,0
2,2019-08-19 13:41:31,2019-08-19 13:41:31,192.168.0.20,192.168.0.2,0,63774,502,10,8,18,...,24,20,44,0.000690,0.051793,0,0.000000,0.0,normal,0
3,2019-08-19 12:43:19,2019-08-19 12:43:20,209.240.235.92,192.168.0.2,0,61771,80,4,0,4,...,0,0,0,0.000000,0.889555,0,419.338813,0.0,DoS,1
4,2019-08-19 14:49:44,2019-08-19 14:49:48,192.168.0.20,192.168.0.1,3,0,0,14,0,14,...,476,0,476,0.000000,3.500055,0,525.146562,0.0,normal,0


In [4]:
df.isna().sum()

StartTime     0
LastTime      0
SrcAddr       0
DstAddr       0
Mean          0
Sport         0
Dport         0
SrcPkts       0
DstPkts       0
TotPkts       0
DstBytes      0
SrcBytes      0
TotBytes      0
SrcLoad       0
DstLoad       0
Load          0
SrcRate       0
DstRate       0
Rate          0
SrcLoss       0
DstLoss       0
Loss          0
pLoss         0
SrcJitter     0
DstJitter     0
SIntPkt       0
DIntPkt       0
Proto         0
Dur           0
TcpRtt        0
IdleTime      0
Sum           0
Min           0
Max           0
sDSb          0
sTtl          0
dTtl          0
sIpId         0
dIpId         0
SAppBytes     0
DAppBytes     0
TotAppByte    0
SynAck        0
RunTime       0
sTos          0
SrcJitAct     0
DstJitAct     0
Traffic       0
Target        0
dtype: int64

Data does not require any filling of null values

In [5]:
# majority of source addresses come from a few IP addresses
df['SrcAddr'].value_counts()

SrcAddr
192.168.0.20                 1090574
209.240.235.92                 46615
192.168.0.10                   26797
192.168.0.44                    8317
20.1.249.77                     7220
49.48.134.64                    6320
192.168.0.2                     4397
01:80:c2:00:00:0e               2529
0.0.0.0                          573
0                                423
fe80::e9ed:931f:c2e0:1333        354
fe80::9bc:3b2b:78d3:855c         321
fe80::dacb:8aff:fe08:ed2a         16
192.168.0.4                        8
Name: count, dtype: int64

In [6]:
df['DstAddr'].value_counts()

DstAddr
192.168.0.2          1155810
192.168.0.20           19765
192.168.0.1             6938
192.168.0.255           3876
6c:b0:ce:22:b0:57       2529
                      ...   
6176                       1
4926                       1
53329                      1
5005                       1
30165                      1
Name: count, Length: 132, dtype: int64

In [7]:
df['Traffic'].value_counts()

Traffic
normal      1107448
DoS           78305
Reconn         8240
CommInj         259
Backdoor        212
Name: count, dtype: int64

In [8]:
df['Target'].value_counts()

Target
0    1107448
1      87016
Name: count, dtype: int64

We can consider either classifying the type of traffic directly, or a 2 step process of:

1. identifying if some traffic is normal or not (using `Target`)
2. Classifying the abnormal traffic into DoS, Reconn, CommInj or Backdoor

In [9]:
df.columns

Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='str')

In [10]:
df[df['SrcAddr'] == '192.168.0.20']

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 12:23:28,2019-08-19 12:23:28,192.168.0.20,192.168.0.2,0,59034,502,10,8,18,...,24,20,44,0.001176,0.053037,0,0.000000,0.0,normal,0
1,2019-08-19 15:13:24,2019-08-19 15:13:24,192.168.0.20,192.168.0.2,0,55841,502,10,8,18,...,24,20,44,0.001308,0.052681,0,0.000000,0.0,normal,0
2,2019-08-19 13:41:31,2019-08-19 13:41:31,192.168.0.20,192.168.0.2,0,63774,502,10,8,18,...,24,20,44,0.000690,0.051793,0,0.000000,0.0,normal,0
4,2019-08-19 14:49:44,2019-08-19 14:49:48,192.168.0.20,192.168.0.1,3,0,0,14,0,14,...,476,0,476,0.000000,3.500055,0,525.146562,0.0,normal,0
6,2019-08-19 11:31:53,2019-08-19 11:31:53,192.168.0.20,192.168.0.2,0,61906,502,6,6,12,...,24,20,44,0.000659,0.029754,0,0.000000,0.0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1194459,2019-08-19 16:41:47,2019-08-19 16:41:47,192.168.0.20,192.168.0.2,0,57730,502,10,8,18,...,24,20,44,0.001238,0.053972,0,0.000000,0.0,normal,0
1194460,2019-08-19 14:39:47,2019-08-19 14:39:47,192.168.0.20,192.168.0.2,0,59930,502,6,6,12,...,24,22,46,0.000646,0.032614,0,0.000000,0.0,normal,0
1194461,2019-08-19 14:06:18,2019-08-19 14:06:18,192.168.0.20,192.168.0.2,0,64384,502,10,8,18,...,24,20,44,0.000668,0.052755,0,0.000000,0.0,normal,0
1194462,2019-08-19 13:20:52,2019-08-19 13:20:52,192.168.0.20,192.168.0.2,0,53833,502,10,8,18,...,24,20,44,0.000676,0.051783,0,0.000000,0.0,normal,0


In [11]:
df = (
    df.sort_values(by=['SrcAddr', 'DstAddr', 'StartTime', 'LastTime'])
        .reset_index(drop=True)
)

In [12]:
# check if there are rows that suddenly turn from normal traffic into abnormal traffic

df_test = df.copy()

df_test['prev_target'] = df_test.groupby(['SrcAddr', 'DstAddr'])['Target'].shift()

df_test['target_changed'] = (
    df_test['prev_target'].notna() &
    df_test['prev_target'].eq('Normal') &
    df_test['Target'].ne('Normal')
)

df_test[df_test['target_changed']]

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target,prev_target,target_changed


In [13]:
df[(df['SrcAddr'] == '0') & (df['DstAddr'] == '14740')]

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
2,2019-08-19 11:05:18,2019-08-19 11:04:18,0,14740,0,0,2,90864,11501,90864,...,14740,0,2019535332,0.0,0.0,0,0.0,0.0,normal,0


In [ ]:
# want to predict whether a particular connection is good or not
# groupby srcaddr and sort by start and end time
# then for each group's row we check if it is anomalous or not?

# maybe the plan could be traditional models -> think time dependency plays a part so use sequence models
# -> think interactions between ip addresses play a part so use graph neural networks

To apply sequence models to this dataset, we group by each source address, sort by start and end time, then create a sliding window across each group, having the next row as the one to predict

In [16]:
df['SrcAddr'].value_counts()

SrcAddr
192.168.0.20                 1090574
209.240.235.92                 46615
192.168.0.10                   26797
192.168.0.44                    8317
20.1.249.77                     7220
49.48.134.64                    6320
192.168.0.2                     4397
01:80:c2:00:00:0e               2529
0.0.0.0                          573
0                                423
fe80::e9ed:931f:c2e0:1333        354
fe80::9bc:3b2b:78d3:855c         321
fe80::dacb:8aff:fe08:ed2a         16
192.168.0.4                        8
Name: count, dtype: int64

In [ ]:
# to find the source address with the smallest window
# decide on the maximum sliding window length available to us

df['SrcAddr'].value_counts().min()

np.int64(8)